la probleme reside dans la problematique suivante dans quelle classe avait on le plus de chance de suvivre

In [155]:
#importation des bibliothèques nécessaires pour le machin learning pour lire et analyser les données 
import pandas as pd
import numpy as np  
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split

chargement du dataset titanic

In [156]:
dataset_titanic = pd.read_csv("data/Titanic-Dataset.csv")

apperçu du dataset titanic des 5 premières lignes

In [157]:
dataset_titanic.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


Afficher les dimensions du dataset titanic

In [158]:
dataset_titanic.shape

(891, 12)

Affichage des noms et type des variables du dataset titanic

In [159]:
dataset_titanic.dtypes

PassengerId      int64
Survived         int64
Pclass           int64
Name               str
Sex                str
Age            float64
SibSp            int64
Parch            int64
Ticket             str
Fare           float64
Cabin              str
Embarked           str
dtype: object

Identification des valeur qui ne sont pas des valeurs numériques dans le dataset titanic

In [160]:
dataset_titanic.select_dtypes(exclude=[np.number])

,Name,Sex,Ticket,Cabin,Embarked
0,"Braund, Mr. Owen Harris",male,A/5 21171,NaN,S
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,PC 17599,C85,C
2,"Heikkinen, Miss. Laina",female,STON/O2. 3101282,NaN,S
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,113803,C123,S
4,"Allen, Mr. William Henry",male,373450,NaN,S
...,...,...,...,...,...
886,"Montvila, Rev. Juozas",male,211536,NaN,S
887,"Graham, Miss. Margaret Edith",female,112053,B42,S
888,"Johnston, Miss. Catherine Helen ""Carrie""",female,W./C. 6607,NaN,S
889,"Behr, Mr. Karl Howell",male,111369,C148,C


remplacement du sex par des valeurs numériques 0 pour male et 1 pour female

In [161]:
dataset_titanic['Sex'] = dataset_titanic['Sex'].map({'male': 0, 'female': 1})
dataset_titanic['Embarked'] = dataset_titanic['Embarked'].map({'C': 0, 'Q': 1, 'S':2})


Supprimer la colonne "Name" du dataset titanic car elle n'est pas utile pour notre analyse

In [162]:
dataset_titanic.drop(columns=['Name'], inplace=True)

supprimer la colonne "Ticket" du dataset titanic car elle n'est pas utile pour notre analyse

In [163]:
dataset_titanic.drop(columns=['Ticket'], inplace=True)

supprimer la colonne "Cabin" du dataset titanic car elle n'est pas utile pour notre analyse

In [164]:
dataset_titanic.drop(columns=['Cabin'], inplace=True)

la variable cible est la variable "Survived" (y) qui indique si le passager a survécu ou non. Nous allons donc la séparer des autres variables pour l'analyse

supprimons les variables inutiles pour notre analyse, à savoir "PassengerId" et "Embarked", 'SibSp', 'Parch' les autres ont ete supprimer plus haut

In [165]:
dataset_titanic.drop(columns=['PassengerId', 'SibSp', 'Parch'], inplace=True)

In [166]:
data_set_utile = dataset_titanic
data_set_utile

,Survived,Pclass,Sex,Age,Fare,Embarked
0,0,3,0,22.0,7.2500,2.0
1,1,1,1,38.0,71.2833,0.0
2,1,3,1,26.0,7.9250,2.0
3,1,1,1,35.0,53.1000,2.0
4,0,3,0,35.0,8.0500,2.0
...,...,...,...,...,...,...
886,0,2,0,27.0,13.0000,2.0
887,1,1,1,19.0,30.0000,2.0
888,0,3,1,NaN,23.4500,2.0
889,1,1,0,26.0,30.0000,0.0


In [167]:
data_set_utile.shape

(891, 6)

detection des valeurs manquantes dans le dataset titanic

In [168]:
data_set_utile.isnull().sum()

Survived      0
Pclass        0
Sex           0
Age         177
Fare          0
Embarked      2
dtype: int64

pourcentage de valeurs manquantes dans le dataset titanic

In [169]:
data_set_utile.isnull().mean() * 100

Survived     0.000000
Pclass       0.000000
Sex          0.000000
Age         19.865320
Fare         0.000000
Embarked     0.224467
dtype: float64

Comparaison entre l'age et le prix du billet pour voir s'il y a une relation entre ces deux variables sous frone de tableau avec les moyennes de prix des billets par age

In [170]:
# Corrélation âge-prix (utiliser df_clean, pas df)
print('Corrélation âge-prix :', round(data_set_utile['Age'].corr(data_set_utile['Fare']), 3))

# Créer les tranches d'âge
data_set_utile['tranche_age'] = pd.cut(
    data_set_utile['Age'],
    bins=[0, 10, 20, 30, 40, 50, 60, 70, 80, 100]
)

# Prix moyen et nombre de passagers par tranche d'âge
print('\nPrix moyen du billet par tranche d\'âge :')
print(
    data_set_utile.groupby('tranche_age', observed=True)['Fare']
    .agg(['mean', 'count'])
    .round(2)
    .rename(columns={'mean': 'prix_moyen', 'count': 'nombre'})
)

Corrélation âge-prix : 0.096

Prix moyen du billet par tranche d'âge :
             prix_moyen  nombre
tranche_age                    
(0, 10]           30.43      64
(10, 20]          29.53     115
(20, 30]          28.31     230
(30, 40]          42.50     155
(40, 50]          41.16      86
(50, 60]          44.77      42
(60, 70]          45.91      17
(70, 80]          25.94       5


remplacer les age manquants par la moyenne des tranche d'age de la classe

In [171]:
# Remplacer les âges manquants par la moyenne d'âge de leur classe (Pclass)
data_set_utile['Age'] = data_set_utile['Age'].fillna(data_set_utile.groupby('Pclass')['Age'].transform('mean'))
# Supprimer les lignes où la colonne "Embarked" est vide
data_set_utile = data_set_utile.dropna(subset=["Embarked"])

In [172]:
data_set_utile

,Survived,Pclass,Sex,Age,Fare,Embarked,tranche_age
0,0,3,0,22.00000,7.2500,2.0,"(20.0, 30.0]"
1,1,1,1,38.00000,71.2833,0.0,"(30.0, 40.0]"
2,1,3,1,26.00000,7.9250,2.0,"(20.0, 30.0]"
3,1,1,1,35.00000,53.1000,2.0,"(30.0, 40.0]"
4,0,3,0,35.00000,8.0500,2.0,"(30.0, 40.0]"
...,...,...,...,...,...,...,...
886,0,2,0,27.00000,13.0000,2.0,"(20.0, 30.0]"
887,1,1,1,19.00000,30.0000,2.0,"(10.0, 20.0]"
888,0,3,1,25.14062,23.4500,2.0,NaN
889,1,1,0,26.00000,30.0000,0.0,"(20.0, 30.0]"


In [173]:
data_set_utile.isnull().sum()

Survived         0
Pclass           0
Sex              0
Age              0
Fare             0
Embarked         0
tranche_age    177
dtype: int64

In [174]:
data_set_utile.drop(columns=['tranche_age'], inplace=True)
data_set_utile.isnull().sum()

Survived    0
Pclass      0
Sex         0
Age         0
Fare        0
Embarked    0
dtype: int64

Tous ce qui a ete fais plus haut est elementaire et absolument neccessaire pour mettre nos donnees au propre et pourvoir maintenant entrainer le modele

probleme de classification lineaire a 2 classe car nous avons que suvecu (0 ou 1)

choix du modele KNN a utiliser

divisons le y et x

In [175]:
y = data_set_utile['Survived']
X = data_set_utile.drop(columns=['Survived'])
X.head()

,Pclass,Sex,Age,Fare,Embarked
0,3,0,22.0,7.2500,2.0
1,1,1,38.0,71.2833,0.0
2,3,1,26.0,7.9250,2.0
3,1,1,35.0,53.1000,2.0
4,3,0,35.0,8.0500,2.0


tout d'abord nous allons separer les donnees de l'entrainnement et les donnees de test dans un pourcentage de 80% et 20 %

In [176]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,       # 20 % pour le test, 80 % pour l'entraînement
    random_state=42,     # résultat reproductible
    stratify=y           # garde la même proportion de suvivant dans les deux ensembles
)

appel au modele KNN et ses hyperparametres

In [177]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

knn = make_pipeline( StandardScaler(), KNeighborsClassifier(n_neighbors=5, weights="distance", algorithm="auto",
                     leaf_size=30, p=2, metric="minkowski",
                     metric_params=None, n_jobs=None) ) # p=2 correspond à la distance euclidienne

entrainement du modele KNN sur les donnees d'entrainement

In [178]:
knn.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('standardscaler', ...), ('kneighborsclassifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](5,)","['Pclass','Sex','Age','Fare','Embarked']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,5
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


prediction sur les donnees de test

In [179]:
y_pred = knn.predict(X_test)

affichage de la matrice de confusion et du score de precision du modele KNN

In [180]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score


print("Score F1 :", f1_score(y_test, y_pred)*100)
# équivalent : print(knn.score(X_test, y_test))

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

Score F1 : 76.33587786259542
[[97 13]
 [18 50]]
              precision    recall  f1-score   support

           0       0.84      0.88      0.86       110
           1       0.79      0.74      0.76        68

    accuracy                           0.83       178
   macro avg       0.82      0.81      0.81       178
weighted avg       0.82      0.83      0.82       178



In [181]:
nouveau_passager = pd.DataFrame([{
    "Pclass": 3,
    "Sex": 1,
    "Age": 75,
    "Fare": 25.94,
    "Embarked" : 0
}])

# Même ordre de colonnes que pendant l'entraînement
nouveau_passager = nouveau_passager[X_train.columns]

# Prédiction
prediction = knn.predict(nouveau_passager)
proba = knn.predict_proba(nouveau_passager)

print("Résultat :", "Survit" if prediction[0] == 1 else "Ne survit pas")
print(f"Probabilité de survie : {proba[0][1]:.2%}")

Résultat : Ne survit pas
Probabilité de survie : 36.92%


le deploiement des modeles de machin learning reposent sur 4 questions fondamentales
1) quand la prediction est elle necessaire (immediatement, hebdomadaire, semitrielle, **en continu**) ?
2) où doit elle etre calculée (sur le poste de travail utilisateur, sur un serveur, dans le cloud) ?
3) Combien de requettes et quelle latence maximale ?
4) est ce que le systeme est securisé et respect la reglementation ?

ces 4 questions renvoient a 3 axe de recherche fondamentale pour le deploiement des modeles de machin learning
1) le modele de prediction : comment les modeles sont ils servit
2) le lieu d'hebergement 
3) la forme du packaging

In [187]:
import zipfile
import os
import joblib

# 4. Sérialisation du pipeline complet (StandardScaler + KNN)
# Modèle sérialisé avec compression (niveau de 0 à 9)
joblib.dump(knn, "modele_titanic.joblib", compress=3)

# 5. Vérification que le fichier a bien été créé
if os.path.exists("modele_titanic.joblib"):
    taille = os.path.getsize("modele_titanic.joblib") / 1024
    print(f"Fichier créé : modele_titanic.joblib ({taille:.1f} Ko)")
else:
    print("Erreur : le fichier n'a pas été créé")

# compression du modele

# Prédictions AVANT sérialisation, enregistrées pour la comparaison
pred_avant = knn.predict(X_test)
proba_avant = knn.predict_proba(X_test)
np.save("pred_avant.npy", pred_avant)
np.save("proba_avant.npy", proba_avant)

# Prédiction AVANT pour le nouveau passager
nouveau_passager = pd.DataFrame([{"Pclass": 2, "Sex": 1, "Age": 22, "Fare": 28.31, "Embarked" : 0}])[X_train.columns]
print("Avant sérialisation :", knn.predict(nouveau_passager)[0],
      knn.predict_proba(nouveau_passager)[0])

Fichier créé : modele_titanic.joblib (10.9 Ko)
Avant sérialisation : 1 [0.11903162 0.88096838]


In [183]:
#Comparaison avant et après (étape 8)
modele_recharge = joblib.load("modele_titanic.joblib")

pred_apres = modele_recharge.predict(X_test)
proba_apres = modele_recharge.predict_proba(X_test)

print("Prédictions identiques :", np.array_equal(pred_avant, pred_apres))
print("Probabilités identiques :", np.allclose(proba_avant, proba_apres))

Prédictions identiques : True
Probabilités identiques : True


9. Intérêt de la sérialisation pour le déploiement
* Entraîner une fois, utiliser partout : l’entraînement peut être long et dépend des données d’origine. Une fois le modèle sauvegardé, on le recharge en quelques millisecondes sans réentraîner.
* Séparation entraînement et production : le data scientist entraîne le modèle dans un notebook, puis le serveur (API Flask, FastAPI, application web) charge simplement le fichier pour prédire.
* Reproductibilité : le même fichier donne les mêmes prédictions, ce qui permet de tester, versionner et partager le modèle.
* Économie de ressources : pas besoin de conserver ni de recharger le jeu d’entraînement et le code de préparation à chaque utilisation. Pour KNN, le fichier contient quand même les données d’entraînement, donc il peut être plus lourd que d’autres modèles.

Axe 2: lieu d'hebergement ou environnement d'hebergement : ici on distingue 5 environnement d'hebergement a savoir :
1) on promise : serveur propre a l'organisation, sur site ou dans un datacenter
2) Cloud public : serveur dans le cloud d'un fournisseur (AWS, Azure, GCP, OVH, etc.)
3) cloud priver : 
4) environnement hybride
5) edge: sur l'appareil de l'utilisateur 

Axe 1 : mode de deploiement ; la litterature (article) presente 5 mode de deploiement a savoir :
1) par lot : ou les outils les plus utiliser sont Apache Airflow, Apache Spark, Talend, Informatica, Microsoft SSIS, Pentaho Data Integration (PDI), Alteryx, Dataiku DSS, AWS Glue, Google Cloud Dataflow, Azure Data Factory; ici le modele s'execute par intervale de temps
2) le mode de deploiement en ligne (temps réel) : ou les outils les plus utiliser sont Flask, FastAPI, Django, Node.js, Spring Boot, TensorFlow Serving, TorchServe, ONNX Runtime, NVIDIA Triton Inference Server; ici le modele s'execute en temps reel
3) le mode de deploiement en flux continu : 
4) mode deploiement embarquer : les outils les plus utiliser sont TensorFlow Lite, ONNX Runtime, PyTorch Mobile, Core ML, Edge Impulse, NVIDIA Jetson, Raspberry Pi; ici le modele s'execute sur l'appareil de l'utilisateur
5) le mode integrer dans l'application

Exposition d'un modele de ML sous forme d'une api , il faut 2 phase; celle d'experimentation et
source de donnees ----> Traitement de donnees -----> modelisation -----> Modele ML
\n regles Metier------|

Exercice : une entreprise soit devlopper un service permettant de definir le risque d'impayer d'un client banquaire a partir de 3 variables :
* le nombre de retard de paiement durant les 12 derniers mois
* le montant de la facture
* l'anciennete du client
question:
1) a partir de ces attribut telecharger un jeu de donnees correspondant
2) definir les regles metieres liees a ce probleme

comment reussit l'industrialisation du modele ML grace au MLOPs
Question:
1) environ 87 des projet ML echoue sur le terrrain